# Recall@10 Evaluation — replay against the 10 official trace files

Ground truth = the final table (end_of_conversation: true) in each trace.
We replay the trace's own scripted user lines against our live /chat,
using OUR agent's real replies as conversation history (not the trace's
canned agent replies), then compare our final recommendations to the
trace's final table.

In [32]:
import re
import glob
import time
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
TRACE_DIR = Path("../../traces")  # C1.md ... C10.md at repo root
REQUEST_TIMEOUT = 60

## Parse a trace .md into (ordered user messages, expected shortlist URLs)

In [33]:
def parse_trace(text: str):
    # Split into turn blocks
    turn_blocks = re.split(r"### Turn \d+", text)[1:]  # drop preamble before first turn

    user_messages = []
    final_urls = []
    final_eoc = False

    for block in turn_blocks:
        # User line: "> quoted text" right after **User**
        user_match = re.search(r"\*\*User\*\*\s*\n+>\s*(.+?)(?:\n\n|\Z)", block, re.DOTALL)
        if user_match:
            user_text = user_match.group(1)
            # Strip leading "> " continuation markers on multi-line quotes
            user_text = re.sub(r"^\>\s*", "", user_text, flags=re.MULTILINE).strip()
            user_messages.append(user_text)

        # end_of_conversation flag for this turn
        eoc_match = re.search(r"end_of_conversation.*?\*\*(true|false)\*\*", block, re.IGNORECASE)
        is_final = eoc_match and eoc_match.group(1).lower() == "true"

        # Table rows: markdown table lines, extract URL column (last | ... |)
        table_rows = re.findall(r"\|\s*\d+\s*\|.+?<(https://[^\s|>]+)>\s*\|", block)

        if is_final and table_rows:
            final_urls = table_rows
            final_eoc = True

    return user_messages, final_urls, final_eoc


def normalize_url(url: str) -> str:
    return url.rstrip("/").replace("/solutions", "").lower()

## Load all traces

In [49]:
# %% [markdown]
# ## Select which traces to run this batch
# Run in batches to manage API quota — e.g. traces[0:4], then [4:8], then [8:10]

# %%
BATCH_START = 8  # change this per run
BATCH_END = 10   # change this per run

trace_files = sorted(glob.glob(f"{TRACE_DIR}/*.md"))
trace_files_batch = trace_files[BATCH_START:BATCH_END]
print(f"Running batch: traces {BATCH_START}-{BATCH_END}")
print(trace_files_batch)

# %%
traces = []
for path in trace_files_batch:
    with open(path, encoding="utf-8") as f:
        text = f.read()
    user_msgs, expected_urls, has_final = parse_trace(text)
    if not has_final or not expected_urls:
        print(f"⚠️  {path}: no final confirmed shortlist found — skipping")
        continue
    traces.append({
        "file": path,
        "user_messages": user_msgs,
        "expected_urls": [normalize_url(u) for u in expected_urls],
    })
    print(f"{path}: {len(user_msgs)} user turns, {len(expected_urls)} expected items")

Running batch: traces 8-10
['..\\..\\traces\\C8.md', '..\\..\\traces\\C9.md']
..\..\traces\C8.md: 3 user turns, 5 expected items
..\..\traces\C9.md: 7 user turns, 7 expected items


## Replay a trace against the live /chat endpoint

Uses the trace's own scripted user lines, but OUR agent's real replies
as assistant history (the trace's original agent text is only used as
ground truth for the final list, never fed back in).

In [50]:
def replay_trace(user_messages, base_url=BASE_URL, max_extra_rounds=2):
    history = []
    last_recs = None
    last_eoc = False

    all_lines = list(user_messages)
    round_count = 0

    while all_lines or (not last_eoc and round_count < len(user_messages) + max_extra_rounds):
        if all_lines:
            next_user_line = all_lines.pop(0)
        else:
            # Our agent asked something the trace didn't anticipate — end gracefully
            break

        history.append({"role": "user", "content": next_user_line})

        for attempt in range(3):
            resp = requests.post(f"{base_url}/chat", json={"messages": history}, timeout=REQUEST_TIMEOUT)
            if resp.status_code in (429, 500, 503):
                wait = 20 * (attempt + 1)
                print(f"    retrying after {wait}s (status {resp.status_code})")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            break
        result = resp.json()

        history.append({"role": "assistant", "content": result.get("reply", "")})

        if result.get("recommendations"):
            last_recs = result["recommendations"]
        last_eoc = result.get("end_of_conversation", False)

        round_count += 1
        time.sleep(15)  # Gemini free-tier RPM pacing

        if last_eoc:
            break

    return last_recs or [], last_eoc, history

## Recall computation

In [45]:
def recall_at_k(predicted_urls, relevant_urls, k=10):
    predicted = set(predicted_urls[:k])
    relevant = set(relevant_urls)
    if not relevant:
        return 0.0
    return len(predicted & relevant) / len(relevant)

## Run full replay evaluation across all traces

**NOTE:** make sure `uvicorn src.main:app --reload` is running before executing the next cell.

In [51]:
# %% [markdown]
# ## Run full replay evaluation across the selected batch

# %%
results = []
for t in traces:
    print(f"\n=== Replaying {t['file']} ===")
    recs, reached_end, history = replay_trace(t["user_messages"])
    predicted_urls = [normalize_url(r.get("url", "")) for r in recs]

    r = recall_at_k(predicted_urls, t["expected_urls"], k=10)
    results.append({
        "trace": t["file"],
        "recall": r,
        "reached_end_of_conversation": reached_end,
        "num_predicted": len(predicted_urls),
        "num_expected": len(t["expected_urls"]),
        "predicted_urls": predicted_urls,
        "expected_urls": t["expected_urls"],
        "history": history,
    })
    print(f"  Recall@10: {r:.2f}  (reached end: {reached_end})")

# %%
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"BATCH {BATCH_START}-{BATCH_END} MEAN RECALL@10: {results_df['recall'].mean():.4f}")
print(f"{'='*60}")
results_df[["trace", "recall", "reached_end_of_conversation", "num_predicted", "num_expected"]]

# %% [markdown]
# ## Save this batch's results to disk (so batches don't overwrite each other)

# %%
import pickle

with open(f"eval_results_batch_{BATCH_START}_{BATCH_END}.pkl", "wb") as f:
    pickle.dump(results, f)
print(f"Saved -> eval_results_batch_{BATCH_START}_{BATCH_END}.pkl")

# %% [markdown]
# ## Diagnostic: traces that didn't fully converge

# %%
weak = results_df[results_df["recall"] < 1.0]
for _, row in weak.iterrows():
    print(f"\n{row['trace']}  (recall={row['recall']:.2f})")
    print(f"  Predicted: {row['predicted_urls']}")
    print(f"  Expected:  {row['expected_urls']}")


=== Replaying ..\..\traces\C8.md ===


  Recall@10: 0.00  (reached end: True)

=== Replaying ..\..\traces\C9.md ===
  Recall@10: 0.14  (reached end: True)

BATCH 8-10 MEAN RECALL@10: 0.0714
Saved -> eval_results_batch_8_10.pkl

..\..\traces\C8.md  (recall=0.00)
  Predicted: ['https://www.shl.com/products/product-catalog/view/microsoft-word-365-essentials-new', 'https://www.shl.com/products/product-catalog/view/microsoft-excel-365-essentials-new', 'https://www.shl.com/products/product-catalog/view/ms-office-basic-computer-literacy-sim-new']
  Expected:  ['https://www.shl.com/products/product-catalog/view/microsoft-excel-365-new', 'https://www.shl.com/products/product-catalog/view/microsoft-word-365-new', 'https://www.shl.com/products/product-catalog/view/ms-excel-new', 'https://www.shl.com/products/product-catalog/view/ms-word-new', 'https://www.shl.com/products/product-catalog/view/occupational-personality-questionnaire-opq32r']

..\..\traces\C9.md  (recall=0.14)
  Predicted: ['https://www.shl.com/products/product-catalog/v

In [56]:
# %% [markdown]
# ## Combine all batches into final Mean Recall@10 (run only after all batches complete)

# %%
import glob

all_results = []
for f in glob.glob("eval_results_batch_*.pkl"):
    with open(f, "rb") as fh:
        all_results.extend(pickle.load(fh))

all_results_df = pd.DataFrame(all_results)
print(f"Total traces evaluated: {len(all_results_df)}")
print(f"\n{'='*60}")
print(f"FINAL MEAN RECALL@10 (all traces): {all_results_df['recall'].mean():.4f}")
print(f"{'='*60}")
all_results_df[["trace", "recall", "reached_end_of_conversation"]]

Total traces evaluated: 9

FINAL MEAN RECALL@10 (all traces): 0.4140


,trace,recall,reached_end_of_conversation
0,..\..\traces\C1.md,0.333333,True
1,..\..\traces\C10.md,1.000000,True
2,..\..\traces\C2.md,0.200000,False
3,..\..\traces\C3.md,0.750000,True
4,..\..\traces\C4.md,0.600000,True
5,..\..\traces\C6.md,0.500000,True
6,..\..\traces\C7.md,0.200000,True
7,..\..\traces\C8.md,0.000000,True
8,..\..\traces\C9.md,0.142857,True


## Diagnostic: traces that didn't fully converge

In [48]:
weak = results_df[results_df["recall"] < 1.0]
for _, row in weak.iterrows():
    print(f"\n{row['trace']}  (recall={row['recall']:.2f})")
    print(f"  Predicted: {row['predicted_urls']}")
    print(f"  Expected:  {row['expected_urls']}")


..\..\traces\C6.md  (recall=0.50)
  Predicted: ['https://www.shl.com/products/product-catalog/view/safety-and-dependability-focus-8-0', 'https://www.shl.com/products/product-catalog/view/essential-focus-8-0', 'https://www.shl.com/products/product-catalog/view/vigilance-focus-8-0']
  Expected:  ['https://www.shl.com/products/product-catalog/view/safety-and-dependability-focus-8-0', 'https://www.shl.com/products/product-catalog/view/workplace-health-and-safety-new']

..\..\traces\C7.md  (recall=0.20)
  Predicted: ['https://www.shl.com/products/product-catalog/view/hipaa-security', 'https://www.shl.com/products/product-catalog/view/svar-spoken-spanish-north-american-new', 'https://www.shl.com/products/product-catalog/view/smart-interview-live']
  Expected:  ['https://www.shl.com/products/product-catalog/view/hipaa-security', 'https://www.shl.com/products/product-catalog/view/medical-terminology-new', 'https://www.shl.com/products/product-catalog/view/microsoft-word-365-essentials-new', '

In [55]:
ls eval_results_batch_*.pkl


 Volume in drive C is Windows 
 Volume Serial Number is 8477-9453

 Directory of c:\Users\DELL\Documents\GitHub\Recomendation_System\src\Evaluation

07/03/2026  11:05 AM            14,458 eval_results_batch_0_4.pkl
07/03/2026  11:47 AM             5,946 eval_results_batch_4_5.pkl
07/03/2026  11:51 AM             8,135 eval_results_batch_6_8.pkl
07/03/2026  11:57 AM            13,067 eval_results_batch_8_10.pkl
               4 File(s)         41,606 bytes
               0 Dir(s)  253,771,657,216 bytes free
